###Dimension Table (Towers) Load - CDC, SCD1, SCD2, CDF, Reporting

![](devices_cdc.png)

In [0]:
  -- Bronze Table
CREATE TABLE IF NOT EXISTS telecom_bronze.dim_device_bronze (
  device_id INT, device_type STRING, brand STRING, model STRING, os STRING, 
  owner_customer_id INT, status STRING, updated_at TIMESTAMP);
  -- Silver Table
CREATE TABLE IF NOT EXISTS telecom_silver.dim_device_silver (
    device_id STRING, brand STRING, model STRING, os STRING, 
    device_type STRING, owner_customer_id STRING, status STRING, updated_at TIMESTAMP
) USING DELTA;

-- Gold SCD1 Table (CDF Enabled)
CREATE TABLE IF NOT EXISTS telecom_gold.dim_device_gold_scd1 (
    device_id STRING, brand STRING, model STRING, os STRING, 
    device_type STRING, owner_customer_id STRING, status STRING, updated_at TIMESTAMP
) USING DELTA TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Gold SCD2 Table (Includes DLT-style history columns)
CREATE TABLE IF NOT EXISTS telecom_gold.dim_device_gold_scd2 (
    device_id STRING, brand STRING, model STRING, os STRING, 
    device_type STRING, owner_customer_id STRING, status STRING, updated_at TIMESTAMP,
    start_at TIMESTAMP, end_at TIMESTAMP
) USING DELTA;

In [0]:
select * from telecom_bronze.dim_device_bronze limit 10;

In [0]:
%sql
--1. CDC (History load for the first time alone) applying Watermarking Feature using Foreign Catalog
INSERT INTO telecom_bronze.dim_device_bronze
SELECT 
    device_id, 
    device_type, 
    brand, 
    model, 
    os, 
    owner_customer_id, 
    status, 
    updated_at
FROM telecom_foreign_catalog.telco_schema.dim_device
WHERE updated_at > (SELECT COALESCE(MAX(updated_at), '1900-01-01') 
    FROM telecom_bronze.dim_device_bronze);
    --This is checkpointing or watermarking feature to achieve incremental load/CDC

In [0]:
%python
from pyspark.sql.functions import col, upper, trim, when, row_number
from pyspark.sql.window import Window

# ====================================================================
# 1. READ BRONZE & LOAD CURATED SILVER (BATCH)
# ====================================================================

raw_df = spark.read.table("telecom_bronze.dim_device_bronze")
# Apply filters and transformations
silver_df = (
    raw_df
    .filter(col("device_id").isNotNull()) 
    .withColumn("device_id", col("device_id").cast("string"))
    .withColumn("owner_customer_id", col("owner_customer_id").cast("string"))
    .withColumn("brand", upper(trim(col("brand"))))
    .withColumn("model", trim(col("model")))
    .withColumn("os", when(col("os").rlike("(?i)^ios$"), "iOS")
                      .otherwise(upper(trim(col("os"))))))

silver_df.write.format("delta").mode("append").saveAsTable("telecom_silver.dim_device_silver")

# ====================================================================
# 2. PREPARE LATEST UPDATES FOR MERGE
# ====================================================================
# Deduplicate to merge only latest updated device

latest_updates_df = (
    silver_df
    .withColumn("rn", row_number().over(Window.partitionBy("device_id").orderBy(col("updated_at").desc())))
    .filter(col("rn") == 1)
    .drop("rn"))

latest_updates_df.createOrReplaceTempView("dedup_latest_silver")

# ====================================================================
# 3. GOLD LAYER: SCD TYPE 1 (Hard Deletes + Hash Comparison)
# ====================================================================
spark.sql("""
    MERGE INTO telecom_gold.dim_device_gold_scd1 t
    USING dedup_latest_silver s
    ON t.device_id = s.device_id
    
    WHEN MATCHED AND s.status = 'Inactive' THEN 
        DELETE -- Hard delete
        
    WHEN MATCHED AND s.updated_at > t.updated_at 
                 AND hash(s.device_type, s.brand, s.model, s.os, s.owner_customer_id, s.status) <> 
                     hash(t.device_type, t.brand, t.model, t.os, t.owner_customer_id, t.status) THEN 
        UPDATE SET *
        
    WHEN NOT MATCHED AND s.status != 'Inactive' THEN 
        INSERT *
""")

# ====================================================================
# 4. GOLD LAYER: SCD TYPE 2 (Hash Comparison)
# ====================================================================
# Create a staging table that marks what needs to be closed and what needs to be inserted
# We use NULL as a merge_key for new records to force the "NOT MATCHED" path
staged_updates = spark.sql("""
    SELECT s.*, s.device_id as merge_key FROM dedup_latest_silver s
    UNION ALL
    SELECT s.*, NULL as merge_key FROM dedup_latest_silver s
    JOIN telecom_gold.dim_device_gold_scd2 t ON s.device_id = t.device_id
    WHERE t.end_at IS NULL 
      AND hash(s.device_type, s.brand, s.model, s.os, s.owner_customer_id, s.status) <> 
          hash(t.device_type, t.brand, t.model, t.os, t.owner_customer_id, t.status)
""")

staged_updates.createOrReplaceTempView("staged_updates")

spark.sql("""
    MERGE INTO telecom_gold.dim_device_gold_scd2 t
    USING staged_updates s
    ON t.device_id = s.merge_key AND t.end_at IS NULL
    
    /* 1. If we matched the merge_key and the hash is different, CLOSE the old record */
    WHEN MATCHED AND hash(s.device_type, s.brand, s.model, s.os, s.owner_customer_id, s.status) <> 
                     hash(t.device_type, t.brand, t.model, t.os, t.owner_customer_id, t.status)
    THEN UPDATE SET t.end_at = s.updated_at
    
    /* 2. If we didn't match (merge_key was NULL or ID is brand new), INSERT the new record */
    WHEN NOT MATCHED THEN
    INSERT (device_id, device_type, brand, model, os, owner_customer_id, status, updated_at, start_at, end_at)
    VALUES (s.device_id, s.device_type, s.brand, s.model, s.os, s.owner_customer_id, s.status, s.updated_at, s.updated_at, NULL)
""")